In [2]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from joblib import dump

from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsRegressor
from sklearn.metrics import accuracy_score
from sklearn.metrics import ConfusionMatrixDisplay
from sklearn.metrics import mean_absolute_error

from sklearn.model_selection import GridSearchCV
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.compose import make_column_transformer
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

In [3]:
wine_train = pd.read_parquet("https://lab.cs307.org/wine/data/wine-train.parquet")
wine_test = pd.read_parquet("https://lab.cs307.org/wine/data/wine-test.parquet")

In [4]:
wine_train

,fixed acidity,volatile acidity,citric acid,residual sugar,chlorides,free sulfur dioxide,total sulfur dioxide,density,pH,sulphates,alcohol,quality,color
0,7.6,0.23,0.64,12.9,0.033,54.0,170.0,0.99800,3.00,0.53,8.8,5,white
1,NaN,0.75,0.01,2.2,0.059,11.0,18.0,0.99242,3.39,0.40,NaN,6,red
2,7.4,0.67,0.12,1.6,0.186,5.0,21.0,0.99600,3.39,0.54,9.5,5,red
3,6.4,0.18,0.74,NaN,0.046,54.0,168.0,0.99780,3.58,0.68,10.1,5,white
4,6.7,0.35,0.32,9.0,0.032,29.0,113.0,0.99188,3.13,0.65,12.9,7,white
...,...,...,...,...,...,...,...,...,...,...,...,...,...
4152,6.6,0.32,0.22,16.7,0.046,38.0,133.0,0.99790,3.22,0.67,10.4,6,white
4153,9.2,0.58,0.20,3.0,0.081,15.0,115.0,0.99800,3.23,0.59,9.5,5,red
4154,8.2,0.60,0.17,2.3,0.072,11.0,73.0,0.99630,3.20,0.45,9.3,5,red
4155,6.5,0.23,0.36,16.3,0.038,43.0,133.0,0.99924,3.26,0.41,8.8,5,white


In [5]:
# create X and y for train
X_train = wine_train.drop("quality", axis=1)
y_train = wine_train["quality"]

# create X and y for test
X_test = wine_test.drop("quality", axis=1)
y_test = wine_test["quality"]

In [6]:
numeric_transformer = make_pipeline(SimpleImputer(strategy = 'mean'), StandardScaler())
categorical_transformer = make_pipeline(SimpleImputer(strategy = "most_frequent"), OneHotEncoder())

numeric_features = X_train.select_dtypes(include = ['int64', 'float64']).columns
categorical_features = X_train.select_dtypes(include = ['object', 'category']).columns

preprocessor = make_column_transformer((numeric_transformer, numeric_features), (categorical_transformer, categorical_features))
preprocessor.fit(X_train)
preprocessor.transform(X_train)

array([[ 2.99142487e-01, -6.75318039e-01,  2.28253546e+00, ...,
        -1.46622571e+00,  0.00000000e+00,  1.00000000e+00],
       [-7.00256891e-16,  2.44884086e+00, -2.16228568e+00, ...,
         0.00000000e+00,  1.00000000e+00,  0.00000000e+00],
       [ 1.41458654e-01,  1.96820103e+00, -1.38620580e+00, ...,
        -8.63495080e-01,  1.00000000e+00,  0.00000000e+00],
       ...,
       [ 7.72193988e-01,  1.54764118e+00, -1.03344222e+00, ...,
        -1.03570383e+00,  1.00000000e+00,  0.00000000e+00],
       [-5.68118598e-01, -6.75318039e-01,  3.07059398e-01, ...,
        -1.46622571e+00,  0.00000000e+00,  1.00000000e+00],
       [-6.46960515e-01, -9.15637955e-01,  2.36506681e-01, ...,
        -6.91286329e-01,  0.00000000e+00,  1.00000000e+00]],
      shape=(4157, 13))

In [7]:
# dT = make_pipeline(preprocessor, DecisionTreeClassifier())
# # dT.get_params().keys() <- because we used a pipeline, there are keys to follow in order to access the parameters
# param_grid = {
#     'decisiontreeclassifier__max_depth' : [1, 3, 5, 7, 9], 
#     'decisiontreeclassifier__min_samples_split' : [2, 4, 6, 8],
#     'decisiontreeclassifier__min_samples_leaf' : [2, 4, 6, 8],
#     'decisiontreeclassifier__criterion' : ['entropy', 'gini']
# }
# model = GridSearchCV(
#     estimator = dT,
#     param_grid = param_grid,
#     cv = 5
# )
# model.fit(X_train, y_train)

In [8]:
knn = make_pipeline(preprocessor, KNeighborsClassifier())
knn.get_params().keys()

param_grid = {
    'kneighborsclassifier__n_neighbors' : [10, 50, 100, 300], 
    'kneighborsclassifier__p' : [1, 2],
    'kneighborsclassifier__weights': ['uniform', 'distance'],
    'kneighborsclassifier__n_jobs' : [1, 5, 10, 25]
}

model2 = GridSearchCV(
    estimator = knn,
    param_grid = param_grid
)

model2.fit(X_train, y_train)


c:\Users\lyons\OneDrive\Desktop\cs307\.venv\Lib\site-packages\sklearn\model_selection\_split.py:811: UserWarning: The least populated class in y has only 2 members, which is less than n_splits=5.
  warnings.warn(


,estimator,Pipeline(step...lassifier())])
,param_grid,"{'kneighborsclassifier__n_jobs': [1, 5, ...], 'kneighborsclassifier__n_neighbors': [10, 50, ...], 'kneighborsclassifier__p': [1, 2], 'kneighborsclassifier__weights': ['uniform', 'distance']}"
,scoring,None
,n_jobs,None
,refit,True
,cv,None
,verbose,0
,pre_dispatch,'2*n_jobs'
,error_score,nan
,return_train_score,False
,transformers,"[('pipeline-1', ...), ('pipeline-2', ...)]"


In [9]:
# knnr = make_pipeline(preprocessor, KNeighborsRegressor(n_neighbors = 50))
# knnr.fit(X_train, y_train)

In [10]:
# mae_dt = mean_absolute_error(y_test, model.predict(X_test))
mae_knn = mean_absolute_error(y_test, model2.predict(X_test))
# mae_knnr = mean_absolute_error(y_test, knnr.predict(X_test))
print(mae_knn)

0.40576923076923077


In [11]:
dump(model2, 'wine.joblib')

['wine.joblib']